# Treewidth - lab

In [1]:
import dimacs

In [4]:
print(dimacs.loadGRGraph("graphtw\\e5.gr"))
print(dimacs.loadDecomposition("graphtw\\e5.tw"))

[set(), {2, 3}, {1, 5}, {1, 5}, set(), {2, 3}, set()]
[<dimacs.Bag object at 0x0000025BC8326EA0>, <dimacs.Bag object at 0x0000025BC6FC0F80>, <dimacs.Bag object at 0x0000025BC8327A40>, <dimacs.Bag object at 0x0000025BC8327560>, <dimacs.Bag object at 0x0000025BC8327C50>]


In [6]:
def checkVC(graph: list[set[int]], allowed_vertices: set[int], vertex_cover: set[int]):
    for i in range(len(graph)):
        graph[i] = graph[i] & allowed_vertices

    for vertex in vertex_cover:
        graph[vertex] = set()
    for i in range(len(graph)):
        graph[i] = graph[i] - vertex_cover

    return all([i == set() for i in graph])

In [7]:
def graphHash(node_num: int, node_set: set[int]):
    return hash((node_num, frozenset(node_set)))

In [ ]:
def min_vc(graph: list[set[int]], y: dimacs.Bag, cost_dict: dict[int, dict[frozenset, int]]):
    for child in y.children:
        min_vc(graph, child, cost_dict)

    y_costs = {}

    # leaf
    if len(y.children) == 0:
        y_costs[frozenset()] = 0

    # introduce
    if len(y.children) == 1 and len(list(y.children)[0].bag) < len(y.bag):
        z = list(y.children)[0]
        for child_conf, child_cost in cost_dict[z.id].items():
            new_vertex = list(y.bag - z.bag)[0]
            new_config = child_conf | new_vertex
            y_costs[new_config] = child_cost + 1

            neighbors_in_bag = graph[new_vertex] & y.bag
            if neighbors_in_bag.issubset(child_conf):
                y_costs[child_conf] = child_cost

    # forget, join